# Step 1 - Target Construction: Brain Atrophy Severity (ICV-adjusted)

Builds five atrophy-severity targets for the BASE cohort (n=10,000) from `BrainVol_BASE.csv`:

- **`target_composite`** - composite across 4 atrophy-relevant ROIs (whole_brain, hippocampus, ventricles, amygdala), each residualized on `BV_icv` only, with `BV_ventricles` adjusted by a two-limit Tobit model (it is double-censored, see below), sign-corrected so higher = more atrophy.
- **`target_hippo`** - hippocampal volume alone, residualized on `BV_icv` only, sign-flipped so higher = more atrophy.
- **`target_wholebrain`** - whole-brain volume alone, residualized on `BV_icv` only, sign-flipped so higher = more atrophy.
- **`target_ventricles`** - ventricular volume alone, Tobit-adjusted for `BV_icv` (same latent residual used in the composite); ventricles *enlarge* with atrophy, so no sign flip.
- **`target_amygdala`** - amygdala volume alone, residualized on `BV_icv` only, sign-flipped so higher = more atrophy.

Brain imaging is used **only** to build these targets - it is not used as a model input feature.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

BRAIN_BASE_PATH = "/mnt/d/Healthcare_AI_Challenge/Healthcare_AI_Challenge_2026/BRAIN_IMAGE/BRAIN_IMAGE_SYNTHETIC/BrainVol_BASE.csv"
CLINICAL_DATA1_PATH = "/mnt/d/Healthcare_AI_Challenge/Healthcare_AI_Challenge_2026/CLINICAL_DATA/BASE_DATA1.txt"

KEY_ROIS = ["BV_icv", "BV_whole_brain", "BV_subctx_hippocampus", "BV_ventricles", "BV_subctx_amygdala"]

brain = pd.read_csv(BRAIN_BASE_PATH)
print(brain.shape)
brain[["T_ID"] + KEY_ROIS].describe()

In [ ]:
clinical1 = pd.read_csv(CLINICAL_DATA1_PATH)[["T_ID", "T_SEX", "T_AGE"]]

df = brain[["T_ID"] + KEY_ROIS].merge(clinical1, on="T_ID", how="inner")
assert len(df) == len(brain), "lost rows on merge - check ID alignment"
df.head()

In [ ]:
plt.scatter(df['T_AGE'], df['BV_ventricles'], alpha=0.1)

## ICV vs. ROI volumes: larger heads have larger brain structures

ICV adjustment assumes every ROI scales with head size. Check it directly: correlation of `BV_icv` with each of the four atrophy ROIs.

- **Raw** Pearson r (plus Spearman, which is more robust for the censored `BV_ventricles`).
- **Within sex**: men have both larger ICV and larger ROIs, so a pooled correlation could be driven by sex alone. A positive correlation within each sex shows the ICV-ROI relationship is not just a sex effect.
- **Partial r | age, sex**: correlation of ICV and ROI after regressing both on age and sex.

Expected direction is **positive for all four** (including ventricles: larger heads have larger ventricles, independent of atrophy).

In [ ]:
from scipy.stats import pearsonr, spearmanr

ICV_ROIS = ["BV_whole_brain", "BV_subctx_hippocampus", "BV_ventricles", "BV_subctx_amygdala"]


def partial_r(x, y, Z):
    Z = np.column_stack([np.ones(len(Z)), Z])
    rx = x - Z @ np.linalg.lstsq(Z, x, rcond=None)[0]
    ry = y - Z @ np.linalg.lstsq(Z, y, rcond=None)[0]
    return pearsonr(rx, ry)


icv_rows = []
for roi in ICV_ROIS:
    r, p = pearsonr(df["BV_icv"], df[roi])
    row = {"roi": roi, "pearson_r": r, "pearson_p": p, "spearman_r": spearmanr(df["BV_icv"], df[roi])[0]}
    for sex_val in sorted(df["T_SEX"].unique()):
        m = df["T_SEX"] == sex_val
        row[f"pearson_r_sex{sex_val}"] = pearsonr(df.loc[m, "BV_icv"], df.loc[m, roi])[0]
    pr, pp = partial_r(df["BV_icv"].values, df[roi].values, df[["T_AGE", "T_SEX"]].values.astype(float))
    row.update(partial_r_age_sex=pr, partial_p=pp)
    icv_rows.append(row)

icv_corr = pd.DataFrame(icv_rows).set_index("roi")
assert (icv_corr[["pearson_r", "partial_r_age_sex"]] > 0).all().all(), "an ROI does not scale positively with ICV"
icv_corr.round(4)

In [ ]:
import os

SUPPLE_FIG_DIR = "/mnt/c/Users/sjm57/Desktop/Healthcare_AI_Challenge_2026/Github2/supple_figures"
os.makedirs(SUPPLE_FIG_DIR, exist_ok=True)
SEX_LABELS = {1: "Male", 2: "Female"}  # BASE_CODEBOOK: T_SEX 1 = male, 2 = female

fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
for ax, roi in zip(axes, ICV_ROIS):
    for sex_val, color in zip(sorted(df["T_SEX"].unique()), ["tab:blue", "tab:orange"]):
        m = df["T_SEX"] == sex_val
        ax.scatter(df.loc[m, "BV_icv"], df.loc[m, roi], s=3, alpha=0.2, c=color, label=SEX_LABELS[sex_val])
    slope, intercept = np.polyfit(df["BV_icv"], df[roi], 1)
    xs = np.linspace(df["BV_icv"].min(), df["BV_icv"].max(), 100)
    ax.plot(xs, intercept + slope * xs, color="black", linewidth=1.5)
    ax.set_title(f"{roi}\nr = {icv_corr.loc[roi, 'pearson_r']:.2f}")
    ax.set_xlabel("BV_icv")
    ax.set_ylabel(roi)
axes[0].legend(markerscale=4)
plt.tight_layout()
fig.savefig(f"{SUPPLE_FIG_DIR}/boi_icv_correlation_scatterplot.png", dpi=300, bbox_inches="tight")
plt.show()

fig = plt.figure(figsize=(6, 5))
sns.heatmap(df[KEY_ROIS].corr(), cmap="coolwarm", center=0, vmin=-1, vmax=1, annot=True, fmt=".2f")
plt.title("Raw brain volume correlation")
plt.tight_layout()
fig.savefig(f"{SUPPLE_FIG_DIR}/boi_icv_correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

## ICV adjustment

`BV_icv` (intracranial volume) is set by skull size and doesn't shrink with atrophy, so it's used only as an adjustment **covariate**, not as one of the composite regions. Each of the 4 atrophy ROIs is residualized on `BV_icv` only. Age is deliberately **not** regressed out - age-related atrophy is the signal the downstream model is meant to predict.

In [ ]:
ATROPHY_ROIS = ["BV_whole_brain", "BV_subctx_hippocampus", "BV_ventricles", "BV_subctx_amygdala"]

covariates = df[["BV_icv"]]

residuals = pd.DataFrame(index=df.index)
for roi in ATROPHY_ROIS:
    reg = LinearRegression().fit(covariates, df[roi])
    residuals[roi + "_resid"] = df[roi] - reg.predict(covariates)

residuals.describe()

## Validating the LR adjustment

The residualization above assumes each ROI is roughly linear in `BV_icv`/`T_SEX` with well-behaved (homoscedastic, ~normal) residuals. Check that per ROI:

- **Homoscedasticity** - residual variance shouldn't depend on the fitted value (Breusch-Pagan test).
- **Normality of residuals** - Shapiro-Wilk + skew/kurtosis. Note this isn't required for OLS point estimates to be unbiased, only for exact SEs/CIs - but since these residuals become a modeling *target*, non-normality driven by a real structural cause (vs. just large-n significance) is worth knowing about.
- **Boundary/censoring artifacts** - fraction of raw values pinned exactly at the sample min/max, which a linear model can't represent.

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy import stats

X_diag = sm.add_constant(covariates.astype(float))

diag_rows = []
for roi in ATROPHY_ROIS:
    y = df[roi]
    fit = sm.OLS(y, X_diag).fit()
    resid = fit.resid
    _, bp_p, _, _ = het_breuschpagan(resid, X_diag)
    _, sh_p = stats.shapiro(resid.sample(min(5000, len(resid)), random_state=0))
    diag_rows.append({
        "roi": roi,
        "R2": fit.rsquared,
        "resid_skew": stats.skew(resid),
        "resid_kurtosis": stats.kurtosis(resid),
        "breusch_pagan_p": bp_p,
        "shapiro_p": sh_p,
        "frac_at_min": (y == y.min()).mean(),
        "frac_at_max": (y == y.max()).mean(),
    })

diag = pd.DataFrame(diag_rows).set_index("roi")
diag

In [ ]:
# BV_ventricles-specific check: is the boundary mass a real biological ceiling/floor,
# or a hard clip applied during data generation? If it's a hard clip, residual = boundary -
# fitted must hold ~exactly (corr = -1, slope = -1) for the boundary-pinned subjects, since
# their "residual" is then just an artifact of where the covariate-predicted value landed
# relative to the clip, not a real deviation.
vent_fit = sm.OLS(df["BV_ventricles"], X_diag).fit()
vent_resid = vent_fit.resid
vent_fitted = vent_fit.fittedvalues
at_max = df["BV_ventricles"] == df["BV_ventricles"].max()
at_min = df["BV_ventricles"] == df["BV_ventricles"].min()

print(f"n at ceiling ({df['BV_ventricles'].max()}): {at_max.sum()} ({at_max.mean():.1%})")
print(f"n at floor ({df['BV_ventricles'].min()}): {at_min.sum()} ({at_min.mean():.1%})")
print("corr(fitted, resid) among ceiling points:", np.corrcoef(vent_fitted[at_max], vent_resid[at_max])[0, 1])
print("corr(fitted, resid) among floor points:  ", np.corrcoef(vent_fitted[at_min], vent_resid[at_min])[0, 1])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df["BV_ventricles"], bins=60)
axes[0].set_title("BV_ventricles raw distribution")
colors = np.where(at_max, "red", np.where(at_min, "orange", "steelblue"))
axes[1].scatter(vent_fitted, vent_resid, s=5, alpha=0.3, c=colors)
axes[1].set_xlabel("fitted")
axes[1].set_ylabel("residual")
axes[1].set_title("Residual vs fitted (red=ceiling, orange=floor)")
plt.tight_layout()

### Finding: `BV_ventricles` is double-censored, not just skewed

A substantial fraction of subjects sit exactly on one of two boundary values (a ceiling and a floor). For them `residual = boundary - fitted` holds essentially exactly - a clipping artifact from the synthetic-data generator, not real signal - so OLS linearity/homoscedasticity/normality fail for this ROI. The other 3 ROIs are genuinely continuous (their small BP/Shapiro p-values reflect the large sample size rather than meaningful skew/kurtosis).

`BV_ventricles` is therefore adjusted with a two-limit Tobit model below.

## Tobit adjustment of `BV_ventricles`

Two-limit (left+right censored) Tobit on const + `BV_icv`, censoring limits at the empirical min and max. For censored subjects, the conditional expectation of the latent (uncensored) value is imputed via the inverse Mills ratio and used as the residual instead of the mechanically-bounded OLS residual. The other 3 ROI residuals are unchanged.

In [ ]:
from scipy.optimize import minimize
from scipy.stats import norm

y_vent = df["BV_ventricles"].values.astype(float)
X_tobit = X_diag.values.astype(float)
L_vent, U_vent = y_vent.min(), y_vent.max()

def tobit_negloglik(params, X, y, L, U):
    beta = params[:-1]
    log_sigma = params[-1]
    sigma = np.exp(log_sigma)
    xb = X @ beta

    left = y <= L
    right = y >= U
    mid = ~left & ~right

    ll = np.empty(y.shape[0])
    resid_mid = (y[mid] - xb[mid]) / sigma
    ll[mid] = -0.5 * np.log(2 * np.pi) - log_sigma - 0.5 * resid_mid ** 2
    ll[left] = norm.logcdf((L - xb[left]) / sigma)
    ll[right] = norm.logsf((U - xb[right]) / sigma)
    return -ll.sum()

# start from the OLS fit already computed above (vent_fit) - a cheap, reasonable starting
# point for beta/sigma; log_sigma keeps sigma > 0 during optimization
beta0 = vent_fit.params.values
sigma0 = np.sqrt(vent_fit.mse_resid)
x0 = np.concatenate([beta0, [np.log(sigma0)]])

# BFGS's finite-difference gradient is numerically unstable here because the intercept
# and the BV_icv coefficient sit on very different scales; Nelder-Mead
# (derivative-free) converges cleanly to the same/better objective value.
tobit_fit = minimize(
    tobit_negloglik, x0, args=(X_tobit, y_vent, L_vent, U_vent),
    method="Nelder-Mead", options={"maxiter": 20000, "xatol": 1e-8, "fatol": 1e-8},
)
assert tobit_fit.success, f"Tobit MLE did not converge: {tobit_fit.message}"

beta_tobit = tobit_fit.x[:-1]
sigma_tobit = np.exp(tobit_fit.x[-1])

print("OLS beta:  ", np.round(beta0, 4), "sigma_ols =", round(sigma0, 4))
print("Tobit beta:", np.round(beta_tobit, 4), "sigma_tobit =", round(sigma_tobit, 4))

In [ ]:
xb_tobit = X_tobit @ beta_tobit
left = y_vent <= L_vent
right = y_vent >= U_vent

E_latent = y_vent.copy()

a = (L_vent - xb_tobit[left]) / sigma_tobit
mills_left = np.exp(norm.logpdf(a) - norm.logcdf(a))
E_latent[left] = xb_tobit[left] - sigma_tobit * mills_left

b = (U_vent - xb_tobit[right]) / sigma_tobit
mills_right = np.exp(norm.logpdf(b) - norm.logsf(b))
E_latent[right] = xb_tobit[right] + sigma_tobit * mills_right

residuals["BV_ventricles_resid_tobit"] = E_latent - xb_tobit

print(f"n left-censored: {left.sum()}, n right-censored: {right.sum()}")
residuals[["BV_ventricles_resid", "BV_ventricles_resid_tobit"]].describe()

### Post-adjustment check: residuals are uncorrelated with ICV

After ICV adjustment the residuals should carry no ICV information (r ~ 0). For the three OLS-residualized ROIs this holds by construction; for the Tobit ventricle residual it is an empirical check.

In [ ]:
post_rows = []
for roi, resid_col in [("BV_whole_brain", "BV_whole_brain_resid"),
                       ("BV_subctx_hippocampus", "BV_subctx_hippocampus_resid"),
                       ("BV_ventricles", "BV_ventricles_resid_tobit"),
                       ("BV_subctx_amygdala", "BV_subctx_amygdala_resid")]:
    post_rows.append({"roi": roi, "residual": resid_col,
                      "r_icv_raw": icv_corr.loc[roi, "pearson_r"],
                      "r_icv_residual": pearsonr(df["BV_icv"], residuals[resid_col])[0]})
pd.DataFrame(post_rows).set_index("roi").round(4)

## Sign correction and composite construction

Whole-brain, hippocampal, and amygdala volumes **decrease** with atrophy; ventricular volume **increases**. The three shrinking regions are sign-flipped so that, after standardizing, higher = more atrophy for all 4 residuals. They are combined via PCA PC1 if it captures >=50% of variance, else a z-score average.

In [ ]:
SIGN = {
    "BV_whole_brain_resid": -1,
    "BV_subctx_hippocampus_resid": -1,
    "BV_ventricles_resid": 1,
    "BV_subctx_amygdala_resid": -1,
}

residuals_tobit = residuals.copy()
residuals_tobit["BV_ventricles_resid"] = residuals["BV_ventricles_resid_tobit"]

signed_z = residuals_tobit[list(SIGN)].copy()
for col, s in SIGN.items():
    signed_z[col] = signed_z[col] * s
signed_z = pd.DataFrame(StandardScaler().fit_transform(signed_z), columns=signed_z.columns, index=signed_z.index)
print(signed_z.corr().round(2))  # all pairwise correlations should be positive

pca = PCA(n_components=len(signed_z.columns))
pcs = pca.fit_transform(signed_z)
explained = pca.explained_variance_ratio_
use_pca = explained[0] >= 0.5
print("\nPC explained variance ratio:", np.round(explained, 3), "-> using", "PC1" if use_pca else "z-avg")

target_composite_continuous = (
    pcs[:, 0] * (1 if pca.components_[0].sum() > 0 else -1) if use_pca else signed_z.mean(axis=1).values
)

# hippocampus-only target, sign-flipped so higher = more atrophy
target_hippo_continuous = -1 * StandardScaler().fit_transform(
    residuals[["BV_subctx_hippocampus_resid"]]
).ravel()

# remaining single-ROI targets: the other 3 composite components, same signed/z-scored residuals
# used in the composite (signed_z: ICV-only, Tobit ventricles, higher = more atrophy)
target_wholebrain_continuous = signed_z["BV_whole_brain_resid"].values
target_ventricles_continuous = signed_z["BV_ventricles_resid"].values
target_amygdala_continuous = signed_z["BV_subctx_amygdala_resid"].values
assert np.allclose(target_hippo_continuous, signed_z["BV_subctx_hippocampus_resid"].values)

In [ ]:
CONT_TARGETS = {
    "composite": target_composite_continuous,
    "hippo": target_hippo_continuous,
    "wholebrain": target_wholebrain_continuous,
    "ventricles": target_ventricles_continuous,
    "amygdala": target_amygdala_continuous,
}
pd.DataFrame(CONT_TARGETS).corr().round(2)

In [ ]:
import os
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 8})

fig, ax = plt.subplots(figsize=(4.5, 3.5))

sc = ax.scatter(
    pcs[:, 0], pcs[:, 1],
    c=df["T_AGE"],
    cmap="viridis",
    s=8,
    alpha=0.5
)

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("Age", fontsize=8)
cbar.ax.tick_params(labelsize=8)

ax.set_xlabel(f"PC1: Composite Atrophy Score ({explained[0]:.1%} var)")
ax.set_ylabel(f"PC2 ({explained[1]:.1%} var)")
ax.tick_params(axis='both', labelsize=8)

plt.tight_layout()

outpath = "/mnt/c/Users/sjm57/Desktop/Healthcare_AI_Challenge_2026/Figures/20261021_Github2/20261011_Fig2_PCA_scatter.png"
os.makedirs(os.path.dirname(outpath), exist_ok=True)

fig.savefig(outpath, format="png", dpi=300)
plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 8})

pc1 = pcs[:, 0]
pc2 = pcs[:, 1]

# 20th and 80th percentile thresholds
q20 = np.percentile(pc1, 20)
q80 = np.percentile(pc1, 80)

# 2x2 grid: left column = hist + scatter, right column = colorbar (top cell hidden)
fig, axes = plt.subplots(
    2, 2,
    figsize=(4.8, 5.5),
    gridspec_kw={'height_ratios': [1, 4], 'width_ratios': [1, 0.04],
                 'hspace': 0.05, 'wspace': 0.05},
)
ax_hist, ax_empty = axes[0]
ax_scatter, cax = axes[1]
ax_empty.axis('off')

# Share x-axis so both panels have identical range
ax_hist.sharex(ax_scatter)

# --------------------
# Top histogram
# --------------------
n, bins, patches = ax_hist.hist(
    pc1,
    bins=50,
    edgecolor='black',
    linewidth=0.5
)

# Color bars by bin center
for patch, left, right in zip(patches, bins[:-1], bins[1:]):
    center = (left + right) / 2
    if center < q20:
        patch.set_facecolor('lightblue')
    elif center > q80:
        patch.set_facecolor('lightcoral')
    else:
        patch.set_facecolor('lightgray')

# Vertical threshold lines
ax_hist.axvline(q20, color='blue', linestyle=':', linewidth=1.5)
ax_hist.axvline(q80, color='red', linestyle=':', linewidth=1)

ax_hist.set_ylabel("Count")
ax_hist.tick_params(axis='both', labelsize=8)
ax_hist.tick_params(axis='x', labelbottom=False)
ax_hist.grid(False)

# --------------------
# Bottom PCA scatter
# --------------------
sc = ax_scatter.scatter(
    pc1, pc2,
    c=df["T_AGE"],
    cmap="viridis",
    s=8,
    alpha=0.5
)

# Colorbar in its own axis -> does not shrink the scatter
cbar = fig.colorbar(sc, cax=cax)
cbar.set_label("Age", fontsize=8)
cbar.ax.tick_params(labelsize=8)

ax_scatter.set_xlabel(f"PC1: Composite Atrophy Score ({explained[0]:.1%} var)")
ax_scatter.set_ylabel(f"PC2 ({explained[1]:.1%} var)")
ax_scatter.tick_params(axis='both', labelsize=8)
ax_scatter.grid(False)

# Same x-range for both panels, padded slightly
pad = 0.02 * (pc1.max() - pc1.min())
ax_scatter.set_xlim(pc1.min() - pad, pc1.max() + pad)

# Align the y-labels of the two panels
fig.align_ylabels([ax_hist, ax_scatter])

outpath = "/mnt/c/Users/sjm57/Desktop/Healthcare_AI_Challenge_2026/Figures/20261021_Github2/20261011_Fig2_PCA_scatter_hist.png"
os.makedirs(os.path.dirname(outpath), exist_ok=True)

fig.savefig(outpath, format="png", dpi=300, bbox_inches="tight")
plt.show()       

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

plt.rcParams.update({'font.size': 8})

pc1 = pcs[:, 0]
pc2 = pcs[:, 1]

# 20th and 80th percentile thresholds
q20 = np.percentile(pc1, 20)
q80 = np.percentile(pc1, 80)

# Group shares
pct_low = (pc1 < q20).mean()
pct_nonhigh = (pc1 <= q80).mean()
pct_high = (pc1 > q80).mean()

colors = {
    'low': {
        'bar': '#9ecae1',
        'box': '#deebf7',
        'edge': '#3182bd'
    },
    'nonhigh': {
        'bar': '#d9d9d9',
        'box': '#f0f0f0',
        'edge': '#737373'
    },
    'high': {
        'bar': '#fc9272',
        'box': '#fee0d2',
        'edge': '#de2d26'
    },
}

# --------------------
# Figure layout
# --------------------
fig, axes = plt.subplots(
    2, 2,
    figsize=(4.8, 6.2),
    gridspec_kw={
        'height_ratios': [1.2, 4],
        'width_ratios': [1, 0.04],
        'hspace': 0.05,
        'wspace': 0.05
    }
)

ax_hist, ax_empty = axes[0]
ax_scatter, cax = axes[1]
ax_empty.axis('off')

# Shared x-axis
ax_hist.sharex(ax_scatter)

pad = 0.02 * (pc1.max() - pc1.min())
xmin = pc1.min() - pad
xmax = pc1.max() + pad

ax_scatter.set_xlim(xmin, xmax)

# --------------------
# Percentile threshold lines
# --------------------
for ax in (ax_hist, ax_scatter):
    ax.axvline(
        q20,
        color=colors['low']['edge'],
        linestyle=':',
        linewidth=1,
        zorder=3
    )
    ax.axvline(
        q80,
        color=colors['high']['edge'],
        linestyle=':',
        linewidth=1,
        zorder=3
    )

# --------------------
# Top histogram
# --------------------
n, bins, patches = ax_hist.hist(
    pc1,
    bins=50,
    edgecolor='black',
    linewidth=0.4,
    zorder=2
)

for patch, left, right in zip(patches, bins[:-1], bins[1:]):
    center = (left + right) / 2

    if center < q20:
        patch.set_facecolor(colors['low']['bar'])
    elif center > q80:
        patch.set_facecolor(colors['high']['bar'])
    else:
        patch.set_facecolor(colors['nonhigh']['bar'])

ax_hist.set_ylabel("Count")
ax_hist.tick_params(axis='both', labelsize=8)
ax_hist.tick_params(axis='x', labelbottom=False)
ax_hist.spines['top'].set_visible(False)
ax_hist.spines['right'].set_visible(False)
ax_hist.grid(False)

# --------------------
# Group boxes above histogram
# --------------------
trans = ax_hist.get_xaxis_transform()

gap = 0.004 * (xmax - xmin)

low_y0, low_y1 = 1.04, 1.32
up_y0, up_y1 = 1.36, 1.64

def group_box(x0, x1, y0, y1, text, key):
    ax_hist.add_patch(
        Rectangle(
            (x0, y0),
            x1 - x0,
            y1 - y0,
            transform=trans,
            clip_on=False,
            facecolor=colors[key]['box'],
            edgecolor=colors[key]['edge'],
            linewidth=0.8
        )
    )

    ax_hist.text(
        (x0 + x1) / 2,
        (y0 + y1) / 2,
        text,
        transform=trans,
        clip_on=False,
        ha='center',
        va='center',
        fontsize=7,
        linespacing=1.15,
        color=colors[key]['edge'],
        fontweight='bold'
    )

# Low-risk: bottom 20%
group_box(
    xmin, q20 - gap,
    low_y0, low_y1,
    f"Low-risk\n{pct_low:.0%}",
    'low'
)

# Non High-risk: bottom 80%, includes low-risk
group_box(
    xmin, q80 - gap,
    up_y0, up_y1,
    f"Non High-risk\n{pct_nonhigh:.0%}",
    'nonhigh'
)

# High-risk: top 20%
group_box(
    q80 + gap, xmax,
    low_y0, up_y1,
    f"High-risk\n{pct_high:.0%}",
    'high'
)

# --------------------
# Bottom PCA scatter
# --------------------
sc = ax_scatter.scatter(
    pc1,
    pc2,
    c=df["T_AGE"],
    cmap="viridis",
    s=8,
    alpha=0.5,
    zorder=2
)

# Colorbar
cbar = fig.colorbar(sc, cax=cax)
cbar.set_label("Age", fontsize=8)
cbar.ax.tick_params(labelsize=8)

ax_scatter.set_xlabel(
    f"PC1: Composite Atrophy Score ({explained[0]:.1%} var)"
)
ax_scatter.set_ylabel(
    f"PC2 ({explained[1]:.1%} var)"
)

ax_scatter.tick_params(axis='both', labelsize=8)
ax_scatter.grid(False)

# Align y labels
fig.align_ylabels([ax_hist, ax_scatter])

# Reserve space for group boxes
fig.subplots_adjust(
    top=0.77,
    bottom=0.10,
    left=0.15,
    right=0.88
)

# --------------------
# Save figure
# --------------------
outpath = "/mnt/c/Users/sjm57/Desktop/Healthcare_AI_Challenge_2026/Figures/20261021_Github2/20261011_Fig2_PCA_scatter_hist.png"
os.makedirs(os.path.dirname(outpath), exist_ok=True)

fig.savefig(
    outpath,
    format="png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

plt.rcParams.update({'font.size': 8})

pc1 = pcs[:, 0]
pc2 = pcs[:, 1]

# 90th percentile threshold
q90 = np.percentile(pc1, 90)

# Group shares
pct_nonhigh = (pc1 <= q90).mean()
pct_high = (pc1 > q90).mean()

colors = {
    'nonhigh': {
        'bar': '#d9d9d9',
        'box': '#f0f0f0',
        'edge': '#737373'
    },
    'high': {
        'bar': '#fc9272',
        'box': '#fee0d2',
        'edge': '#de2d26'
    },
}

# --------------------
# Figure layout
# --------------------
fig, axes = plt.subplots(
    2, 2,
    figsize=(4.8, 6.2),
    gridspec_kw={
        'height_ratios': [1.2, 4],
        'width_ratios': [1, 0.04],
        'hspace': 0.05,
        'wspace': 0.05
    }
)

ax_hist, ax_empty = axes[0]
ax_scatter, cax = axes[1]
ax_empty.axis('off')

# Shared x-axis
ax_hist.sharex(ax_scatter)

pad = 0.02 * (pc1.max() - pc1.min())
xmin = pc1.min() - pad
xmax = pc1.max() + pad

ax_scatter.set_xlim(xmin, xmax)

# --------------------
# Percentile threshold line
# --------------------
for ax in (ax_hist, ax_scatter):
    ax.axvline(
        q90,
        color=colors['high']['edge'],
        linestyle=':',
        linewidth=1,
        zorder=3
    )

# --------------------
# Top histogram
# --------------------
n, bins, patches = ax_hist.hist(
    pc1,
    bins=50,
    edgecolor='black',
    linewidth=0.4,
    zorder=2
)

for patch, left, right in zip(patches, bins[:-1], bins[1:]):
    center = (left + right) / 2

    if center > q90:
        patch.set_facecolor(colors['high']['bar'])
    else:
        patch.set_facecolor(colors['nonhigh']['bar'])

ax_hist.set_ylabel("Count")
ax_hist.tick_params(axis='both', labelsize=8)
ax_hist.tick_params(axis='x', labelbottom=False)
ax_hist.spines['top'].set_visible(False)
ax_hist.spines['right'].set_visible(False)
ax_hist.grid(False)

# --------------------
# Group boxes above histogram
# --------------------
trans = ax_hist.get_xaxis_transform()

gap = 0.004 * (xmax - xmin)

box_y0, box_y1 = 1.04, 1.32

def group_box(x0, x1, y0, y1, text, key):
    ax_hist.add_patch(
        Rectangle(
            (x0, y0),
            x1 - x0,
            y1 - y0,
            transform=trans,
            clip_on=False,
            facecolor=colors[key]['box'],
            edgecolor=colors[key]['edge'],
            linewidth=0.8
        )
    )

    ax_hist.text(
        (x0 + x1) / 2,
        (y0 + y1) / 2,
        text,
        transform=trans,
        clip_on=False,
        ha='center',
        va='center',
        fontsize=7,
        linespacing=1.15,
        color=colors[key]['edge'],
        fontweight='bold'
    )

# Non High-risk: bottom 90%
group_box(
    xmin, q90 - gap,
    box_y0, box_y1,
    f"Non High-risk\n{pct_nonhigh:.0%}",
    'nonhigh'
)

# High-risk: top 10%
group_box(
    q90 + gap, xmax,
    box_y0, box_y1,
    f"High-risk\n{pct_high:.0%}",
    'high'
)

# --------------------
# Bottom PCA scatter
# --------------------
sc = ax_scatter.scatter(
    pc1,
    pc2,
    c=df["T_AGE"],
    cmap="viridis",
    s=8,
    alpha=0.5,
    zorder=2
)

# Colorbar
cbar = fig.colorbar(sc, cax=cax)
cbar.set_label("Age", fontsize=8)
cbar.ax.tick_params(labelsize=8)

ax_scatter.set_xlabel(
    f"PC1: Composite Atrophy Score ({explained[0]:.1%} var)"
)
ax_scatter.set_ylabel(
    f"PC2 ({explained[1]:.1%} var)"
)

ax_scatter.tick_params(axis='both', labelsize=8)
ax_scatter.grid(False)

# Align y labels
fig.align_ylabels([ax_hist, ax_scatter])

# Reserve space for group boxes (one row only)
fig.subplots_adjust(
    top=0.82,
    bottom=0.10,
    left=0.15,
    right=0.88
)

# --------------------
# Save figure
# --------------------
outpath = "/mnt/c/Users/sjm57/Desktop/Healthcare_AI_Challenge_2026/Figures/20261021_Github2/20261011_Fig2_PCA_scatter_hist_top10.png"
os.makedirs(os.path.dirname(outpath), exist_ok=True)

fig.savefig(
    outpath,
    format="png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for sex_val, color in zip(sorted(df["T_SEX"].unique()), ["tab:blue", "tab:orange"]):
    mask = df["T_SEX"] == sex_val
    ax.scatter(pcs[mask, 0], pcs[mask, 1], c=color, s=8, alpha=0.5, label=f"T_SEX={sex_val}")
ax.set_xlabel(f"PC1 ({explained[0]:.1%} var)")
ax.set_ylabel(f"PC2 ({explained[1]:.1%} var)")
ax.set_title("PCA of signed, z-scored ROI residuals, colored by sex")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
pc1 = pcs[:, 0]
threshold = np.percentile(pc1, 80)

plt.hist(pc1, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(threshold, color='red', linestyle=':', linewidth=2,
            label=f'Top 20% boundary: {threshold:.2f}')

plt.xlabel('PC1')
plt.ylabel('Count')
plt.legend()
plt.show()

In [ ]:
loadings = pd.DataFrame(
    pca.components_,
    columns=signed_z.columns,
    index=[f"PC{i+1}" for i in range(pca.components_.shape[0])],
)
print("PCA loadings (rows=components, values=per-ROI weights):")
print(loadings.round(3))

for i in range(pcs.shape[1]):
    top = loadings.iloc[i].abs().sort_values(ascending=False)
    r1, r2 = top.index[0], top.index[1]
    print(f"\nPC{i+1} ({explained[i]:.1%} var) top loadings: "
          f"{r1} ({loadings.iloc[i][r1]:+.2f}), {r2} ({loadings.iloc[i][r2]:+.2f})")

for i in range(pcs.shape[1]):
    age_r = np.corrcoef(pcs[:, i], df["T_AGE"])[0, 1]
    sex_r = np.corrcoef(pcs[:, i], df["T_SEX"])[0, 1]
    print(f"\nPC{i+1}: corr with T_AGE = {age_r:.3f}, corr with T_SEX = {sex_r:.3f}")

## Sanity check

In [ ]:
# Correlation among the five key ROI volumes only (baseline)
plt.figure(figsize=(6, 5))
sns.heatmap(signed_z.corr(), cmap='coolwarm', center=0, vmin=-1, vmax=1, annot=True, fmt='.2f')
plt.title('Z-score correlation')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(CONT_TARGETS), figsize=(5 * len(CONT_TARGETS), 4))
for ax, (name, cont) in zip(axes, CONT_TARGETS.items()):
    ax.hist(cont, bins=100, alpha=0.7)
    ax.axvline(np.mean(cont), linestyle="--", linewidth=2, label=f"Mean = {np.mean(cont):.2f}")
    ax.axvline(np.median(cont), linestyle="--", linewidth=2, color="tab:orange", label=f"Median = {np.median(cont):.2f}")
    ax.set_title(f"target_{name}")
    ax.set_ylabel("Count")
    ax.legend()
plt.tight_layout()
plt.show()

## Binarization and risk classes

Two label families per continuous target, all on the full N = 10,000 (no participant is dropped):

- **Top-X% vs rest** (binary): top 20%, 10%, 5% = 1 (high-risk), everyone else = 0 (non high-risk). Positive rates 0.20 / 0.10 / 0.05 - the screening setting, where the whole population has to be classified.
- **Ordered risk classes** (multiclass): equal-size quantile bins, higher class = more atrophy.
  - `risk5` (quintiles): 5 = top 20%, 4 = 20-40%, 3 = 40-60%, 2 = 60-80%, 1 = bottom 20% (2,000 each).
  - `risk4` (quartiles): 4 = top 25%, 3 = 25-50%, 2 = 50-75%, 1 = bottom 25% (2,500 each).

The top class of `risk5` is the same group as `top20`.

In [ ]:
def top_vs_rest(x, q):
    # top q fraction -> 1, everyone else -> 0
    return (x >= np.quantile(x, 1 - q)).astype(int)


def risk_classes(x, k):
    # k equal-size ordered classes: 1 = least atrophy ... k = most atrophy (top 1/k)
    return np.asarray(pd.qcut(x, k, labels=False)) + 1


TOP_SPLITS = {"top20": 0.20, "top10": 0.10, "top5": 0.05}
RISK_CLASSES = {"risk5": 5, "risk4": 4}

targets = pd.DataFrame({"T_ID": df["T_ID"]})
for name, cont in CONT_TARGETS.items():
    targets[f"target_{name}_cont"] = cont
    for split, q in TOP_SPLITS.items():
        targets[f"target_{name}_{split}"] = top_vs_rest(cont, q)
    for split, k in RISK_CLASSES.items():
        targets[f"target_{name}_{split}"] = risk_classes(cont, k)

binary_cols = [f"target_{name}_{split}" for name in CONT_TARGETS for split in TOP_SPLITS]
risk_cols = [f"target_{name}_{split}" for name in CONT_TARGETS for split in RISK_CLASSES]
display(pd.DataFrame({"n_labeled": targets[binary_cols].notna().sum(), "frac_positive": targets[binary_cols].mean().round(3)}))
pd.DataFrame({col: targets[col].value_counts().sort_index() for col in risk_cols}).T

## Sanity checks

All five continuous targets should correlate **positively** with age (older -> more atrophy) even though age was not regressed out - a check on the sign convention. Every label is defined for all 10,000 participants, the risk classes are equal-sized, and the top `risk5` class is exactly the `top20` group.

In [ ]:
check = targets.merge(df[["T_ID", "T_AGE", "T_SEX"]], on="T_ID")
for col in [f"target_{name}_cont" for name in CONT_TARGETS]:
    age_r = np.corrcoef(check["T_AGE"], check[col])[0, 1]
    sex_r = np.corrcoef(check["T_SEX"], check[col])[0, 1]
    print(f"{col:40s} corr(age)={age_r:.3f}  corr(sex)={sex_r:+.3f}")
    assert age_r > 0, "sign convention looks flipped - investigate"

assert targets.notna().all().all(), "every label is defined for every participant"
for name in CONT_TARGETS:
    for split, k in RISK_CLASSES.items():
        assert (targets[f"target_{name}_{split}"].value_counts() == len(targets) // k).all()
    assert (targets[f"target_{name}_top20"] == (targets[f"target_{name}_risk5"] == 5)).all()
assert len(targets) == 10000 and targets.shape[1] == 1 + 6 * len(CONT_TARGETS)
print("\nall label checks passed")

In [ ]:
OUT_PATH = "/mnt/c/Users/sjm57/Desktop/Healthcare_AI_Challenge_2026/Notebook/20261021_Github2/step1_targets.csv"
targets.to_csv(OUT_PATH, index=False)
SUPPLE_TABLE_DIR = "/mnt/c/Users/sjm57/Desktop/Healthcare_AI_Challenge_2026/Github2/supple_tables"
targets.to_csv(f"{SUPPLE_TABLE_DIR}/targets.csv", index=False)
print("saved:", OUT_PATH, targets.shape)

## PC vs. clinical variable correlation screen

Tests each of PC1-PC3 (from the 4-component PCA of the ICV-only, Tobit-adjusted ROI residuals above) against every baseline clinical variable (`BASE_DATA1/2/3`), using the codebook's authoritative continuous/categorical variable-type label rather than inferring type from the data. Missingness is sentinel-coded (`66666`=not surveyed, `77777`=not applicable, `99999`=unknown/no response) and is masked to `NaN` before testing, with per-variable (pairwise) deletion.

- **Continuous vs PC**: Pearson + Spearman.
- **Binary categorical vs PC**: point-biserial correlation.
- **Multi-level categorical vs PC**: one-way ANOVA (F, eta-squared) + Kruskal-Wallis (H).
- **Multiple testing**: Benjamini-Hochberg FDR correction applied globally across all primary p-values (3 PCs x ~58 variables).

In [ ]:
BASE_DATA2_PATH = "/mnt/d/Healthcare_AI_Challenge/Healthcare_AI_Challenge_2026/CLINICAL_DATA/BASE_DATA2.txt"
BASE_DATA3_PATH = "/mnt/d/Healthcare_AI_Challenge/Healthcare_AI_Challenge_2026/CLINICAL_DATA/BASE_DATA3.txt"
CODEBOOK_PATH = "/mnt/d/Healthcare_AI_Challenge/Healthcare_AI_Challenge_2026/CLINICAL_DATA/BASE_CODEBOOk.xlsx"

base1_full = pd.read_csv(CLINICAL_DATA1_PATH)
base2 = pd.read_csv(BASE_DATA2_PATH)
base3 = pd.read_csv(BASE_DATA3_PATH)

clinical_all = (
    df[["T_ID"]]
    .merge(base1_full, on="T_ID")
    .merge(base2, on=["T_ID", "T_DATA_CLASS"])
    .merge(base3, on=["T_ID", "T_DATA_CLASS"])
)
assert clinical_all["T_ID"].equals(df["T_ID"]) and len(clinical_all) == len(df)

codebook = pd.read_excel(CODEBOOK_PATH, header=2)
codebook.columns = [c.strip().replace("\n", "") if isinstance(c, str) else c for c in codebook.columns]
var_type = codebook.set_index("변수명")["변수타입"].to_dict()  # "연속형" (continuous) / "범주형" (categorical)
var_desc = codebook.set_index("변수명")["변수 설명"].to_dict()

SENTINELS = {66666, 77777, 99999}


def mask_sentinels(s):
    return s.where(~s.isin(SENTINELS)) if pd.api.types.is_numeric_dtype(s) else s

In [ ]:
from scipy.stats import pearsonr, spearmanr, pointbiserialr, f_oneway, kruskal
from statsmodels.stats.multitest import multipletests

pc_cols = {"PC1": pcs[:, 0], "PC2": pcs[:, 1], "PC3": pcs[:, 2]}
clinical_vars = [c for c in clinical_all.columns if c != "T_ID"]

rows, skipped = [], []
for var in clinical_vars:
    vtype = var_type.get(var)
    if vtype not in ("연속형", "범주형"):
        skipped.append((var, "no codebook type"))
        continue

    raw = mask_sentinels(clinical_all[var])
    if vtype == "연속형":
        raw = pd.to_numeric(raw, errors="coerce")

    for pc_name, pc_vals in pc_cols.items():
        s = pd.DataFrame({"pc": pc_vals, "x": raw}).dropna()
        n, nun = len(s), s["x"].nunique()
        if n < 3 or nun < 2:
            if pc_name == "PC1":
                skipped.append((var, f"n={n}, nunique={nun}"))
            continue

        row = {"pc": pc_name, "variable": var, "description": var_desc.get(var), "n": n}
        if vtype == "연속형":
            r, p = pearsonr(s["pc"], s["x"])
            sr, sp = spearmanr(s["pc"], s["x"])
            row.update(var_kind="continuous", test="pearson", stat=r, p_value=p,
                       spearman_r=sr, spearman_p=sp)
        elif nun == 2:
            cats = sorted(s["x"].unique())
            binary = s["x"].map({cats[0]: 0, cats[1]: 1})
            r, p = pointbiserialr(binary, s["pc"])
            row.update(var_kind="binary", test="point_biserial", stat=r, p_value=p)
        else:
            groups = [g["pc"].values for _, g in s.groupby("x")]
            F, p = f_oneway(*groups)
            H, kp = kruskal(*groups)
            grand_mean = s["pc"].mean()
            ss_between = sum(len(g) * (g["pc"].mean() - grand_mean) ** 2 for _, g in s.groupby("x"))
            ss_total = ((s["pc"] - grand_mean) ** 2).sum()
            row.update(var_kind="multi_categorical", test="anova", stat=F, p_value=p,
                       eta_squared=ss_between / ss_total, kruskal_H=H, kruskal_p=kp)
        rows.append(row)

results = pd.DataFrame(rows)
results["p_adj_fdr"] = multipletests(results["p_value"], method="fdr_bh")[1]

print(f"{len(results)} tests across {results['variable'].nunique()} variables x {len(pc_cols)} PCs")
print("Skipped (no codebook type, or insufficient/constant data after sentinel-masking):", skipped)

In [ ]:
sex_specific_vars = ["T_MNSAG", "T_PMYN", "T_PMAG", "T_PREG", "T_FPREGAG"]
print("Note:", sex_specific_vars, "are structurally not-applicable for one sex (coded 77777),",
      "so their n reflects the applicable-sex subset only, not missing data.")

for pc_name in pc_cols:
    print(f"\n=== {pc_name}: top 10 clinical associations by FDR-adjusted p-value ===")
    top = results[results["pc"] == pc_name].sort_values("p_adj_fdr").head(10)
    print(top[["variable", "description", "var_kind", "test", "stat", "p_value", "p_adj_fdr", "n"]].to_string(index=False))

CLINICAL_CORR_OUT_PATH = "/mnt/c/Users/sjm57/Desktop/Healthcare_AI_Challenge_2026/Notebook/20261021_Github2/step1_pc_clinical_correlations.csv"
results.to_csv(CLINICAL_CORR_OUT_PATH, index=False)
print("\nsaved:", CLINICAL_CORR_OUT_PATH, results.shape)